# 02 — Data Cleaning
**Clean and prepare event-level data for downstream analytics.**


## Objective
- Define the purpose of this stage.
- Record key decisions and assumptions.
- Keep outputs reproducible and easy to trace to the previous stage.


## Planned transformations
- Parse and validate timestamps
- Inspect duplicate records
- Handle missing values according to documented rules
- Validate price values
- Separate purchase events where required
- Write reusable cleaned data to Parquet


In [ ]:
# Load the raw dataset / exploration outputs here.
# Keep cleaning rules explicit and reproducible.


## Output
The cleaned event/purchase dataset should be written to a local Parquet path excluded from GitHub.


In [1]:
from pyspark.sql import SparkSession

# Create Spark session
spark = (
    SparkSession.builder
    .appName("ECommerceCustomerBehaviorAnalytics-02-Cleaning")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/28 20:18:00 WARN Utils: Your hostname, Vaishnavis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.102 instead (on interface en0)
26/09/28 20:18:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/vaishnavipoojary/Desktop/ecommerce-customer-behavior-analytics/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/28 20:18:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where

Spark version: 4.2.0


In [2]:
# Load raw e-commerce event data

FILE_PATH = "../data/2019-Oct.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(FILE_PATH)
)

print("Rows:", df.count())
print("Columns:", len(df.columns))

df.printSchema()

[Stage 2:======================================================>  (41 + 2) / 43]

Rows: 42448764
Columns: 9
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [3]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Baseline data-quality check before cleaning

quality_check = df.select(
    spark_sum(when(col("event_time").isNull(), 1).otherwise(0)).alias("null_event_time"),
    spark_sum(when(col("event_type").isNull(), 1).otherwise(0)).alias("null_event_type"),
    spark_sum(when(col("product_id").isNull(), 1).otherwise(0)).alias("null_product_id"),
    spark_sum(when(col("category_id").isNull(), 1).otherwise(0)).alias("null_category_id"),
    spark_sum(when(col("category_code").isNull(), 1).otherwise(0)).alias("null_category_code"),
    spark_sum(when(col("brand").isNull(), 1).otherwise(0)).alias("null_brand"),
    spark_sum(when(col("price").isNull(), 1).otherwise(0)).alias("null_price"),
    spark_sum(when(col("user_id").isNull(), 1).otherwise(0)).alias("null_user_id"),
    spark_sum(when(col("user_session").isNull(), 1).otherwise(0)).alias("null_user_session")
)

quality_check.show()

[Stage 5:======================================================>  (41 + 2) / 43]

+---------------+---------------+---------------+----------------+------------------+----------+----------+------------+-----------------+
|null_event_time|null_event_type|null_product_id|null_category_id|null_category_code|null_brand|null_price|null_user_id|null_user_session|
+---------------+---------------+---------------+----------------+------------------+----------+----------+------------+-----------------+
|              0|              0|              0|               0|          13515609|   6113008|         0|           0|                2|
+---------------+---------------+---------------+----------------+------------------+----------+----------+------------+-----------------+



In [4]:
# Inspect records with missing user_session

df.filter(col("user_session").isNull()).show(truncate=False)

[Stage 11:==================================>                     (11 + 7) / 18]

+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------+
|event_time         |event_type|product_id|category_id        |category_code         |brand  |price |user_id  |user_session|
+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------+
|2019-10-06 19:56:10|cart      |1801723   |2053013554415534427|electronics.video.tv  |tcl    |135.65|557388939|NULL        |
|2019-10-25 16:06:14|cart      |1004767   |2053013555631882655|electronics.smartphone|samsung|246.52|549825742|NULL        |
+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------+



In [5]:
# Remove records with missing user_session

df_clean = df.filter(col("user_session").isNotNull())

print("Rows before cleaning:", df.count())
print("Rows after removing null sessions:", df_clean.count())
print("Rows removed:", df.count() - df_clean.count())

Rows before cleaning: 42448764


Rows after removing null sessions: 42448762


[Stage 21:=====================================================>  (41 + 2) / 43]

Rows removed: 2


In [6]:
# Investigate non-positive prices after session cleaning

df_clean.filter(col("price") <= 0) \
    .groupBy("event_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

[Stage 24:=====================================================>  (41 + 2) / 43]

+----------+-----+
|event_type|count|
+----------+-----+
|      view|68523|
|      cart|  150|
+----------+-----+



In [7]:
# Remove records with non-positive prices

df_clean = df_clean.filter(col("price") > 0)

print("Rows after removing null sessions and non-positive prices:", df_clean.count())
print("Rows removed in total:", df.count() - df_clean.count())

Rows after removing null sessions and non-positive prices: 42380089


[Stage 33:=====================================================>  (41 + 2) / 43]

Rows removed in total: 68675


In [8]:
# Handle missing descriptive attributes

df_clean = (
    df_clean
    .fillna({
        "category_code": "unknown",
        "brand": "unknown"
    })
)

print("Cleaning of category_code and brand completed.")

Cleaning of category_code and brand completed.


In [9]:
# Validate cleaned data

df_clean.select(
    spark_sum(when(col("event_time").isNull(), 1).otherwise(0)).alias("null_event_time"),
    spark_sum(when(col("event_type").isNull(), 1).otherwise(0)).alias("null_event_type"),
    spark_sum(when(col("product_id").isNull(), 1).otherwise(0)).alias("null_product_id"),
    spark_sum(when(col("category_id").isNull(), 1).otherwise(0)).alias("null_category_id"),
    spark_sum(when(col("category_code").isNull(), 1).otherwise(0)).alias("null_category_code"),
    spark_sum(when(col("brand").isNull(), 1).otherwise(0)).alias("null_brand"),
    spark_sum(when(col("price").isNull(), 1).otherwise(0)).alias("null_price"),
    spark_sum(when(col("user_id").isNull(), 1).otherwise(0)).alias("null_user_id"),
    spark_sum(when(col("user_session").isNull(), 1).otherwise(0)).alias("null_user_session")
).show()

[Stage 36:=====================================================>  (41 + 2) / 43]

+---------------+---------------+---------------+----------------+------------------+----------+----------+------------+-----------------+
|null_event_time|null_event_type|null_product_id|null_category_id|null_category_code|null_brand|null_price|null_user_id|null_user_session|
+---------------+---------------+---------------+----------------+------------------+----------+----------+------------+-----------------+
|              0|              0|              0|               0|                 0|         0|         0|           0|                0|
+---------------+---------------+---------------+----------------+------------------+----------+----------+------------+-----------------+



In [10]:
# Validate event types

df_clean.groupBy("event_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

[Stage 39:=====================================================>  (41 + 2) / 43]

+----------+--------+
|event_type|   count|
+----------+--------+
|      view|40710876|
|      cart|  926364|
|  purchase|  742849|
+----------+--------+



In [12]:
from pyspark.sql.functions import count

# Check for repeated purchase records without a full-data distinct()

purchase_duplicates = (
    df_clean
    .filter(col("event_type") == "purchase")
    .groupBy(
        "user_id",
        "user_session",
        "product_id",
        "event_time",
        "price"
    )
    .count()
    .filter(col("count") > 1)
)

print("Repeated purchase groups:", purchase_duplicates.count())

[Stage 48:=====================================================>  (41 + 2) / 43]

Repeated purchase groups: 76


In [13]:
# Final price validity check

df_clean.filter(col("price") <= 0).count()

0

In [14]:
# Save cleaned dataset in Parquet format

CLEANED_PATH = "../data/cleaned_ecommerce_oct.parquet"

df_clean.write \
    .mode("overwrite") \
    .parquet(CLEANED_PATH)

print("Cleaned dataset saved to:", CLEANED_PATH)

26/09/28 20:38:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/28 20:38:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/28 20:38:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/28 20:39:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/28 20:39:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/28 20:39:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/28 20:39:01 WARN MemoryManager: Total allocation exceeds 95.00%

Cleaned dataset saved to: ../data/cleaned_ecommerce_oct.parquet


In [15]:
# Verify cleaned Parquet dataset

cleaned_check = spark.read.parquet(CLEANED_PATH)

print("Rows:", cleaned_check.count())
print("Columns:", len(cleaned_check.columns))

cleaned_check.printSchema()

Rows: 42380089
Columns: 9
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [16]:
# Final cleaning summary

raw_count = df.count()
clean_count = cleaned_check.count()

print("========== CLEANING SUMMARY ==========")
print(f"Raw dataset rows:      {raw_count:,}")
print(f"Cleaned dataset rows:  {clean_count:,}")
print(f"Rows removed:          {raw_count - clean_count:,}")
print(f"Rows retained:         {(clean_count / raw_count) * 100:.2f}%")
print("======================================")

[Stage 61:=====================================================>  (41 + 2) / 43]

========== CLEANING SUMMARY ==========
Raw dataset rows:      42,448,764
Cleaned dataset rows:  42,380,089
Rows removed:          68,675
Rows retained:         99.84%


## Final Cleaning Decisions

The raw October 2019 e-commerce event dataset was cleaned using targeted rules based on the exploratory analysis.

### 1. Missing `user_session`
- 2 records contained a missing `user_session`.
- These records were removed because session-level analysis is part of the project and a valid session identifier is required.
- No other fields in these records were missing.

### 2. Non-positive prices
- 68,673 records had `price <= 0`.
- These records consisted of 68,523 `view` events and 150 `cart` events.
- No purchase event had a non-positive price.
- These records were removed because valid positive prices are required for monetary and revenue-based analysis.

### 3. Missing `category_code`
- Missing category codes were replaced with `"unknown"`.
- Records were retained because `category_code` had substantial missingness and removing these records would unnecessarily discard a large portion of the dataset.

### 4. Missing `brand`
- Missing brand values were replaced with `"unknown"`.
- Records were retained for the same reason: brand is a descriptive attribute and its missingness should not cause otherwise valid events to be discarded.

### 5. Event types
- The cleaned dataset contains only the valid event types:
  - `view`
  - `cart`
  - `purchase`

### 6. Repeated purchase groups
- 76 repeated purchase groups were identified using the available fields:
  `user_id`, `user_session`, `product_id`, `event_time`, and `price`.
- These records were retained because the dataset does not provide an explicit transaction/order identifier.
- Therefore, repeated records cannot be confidently classified as erroneous duplicates.

### Final Dataset

The resulting cleaned dataset contains **42,380,089 records across 9 attributes**, representing **99.84% of the original dataset**.

The cleaned data has been saved in **Parquet format** for efficient downstream processing:

`data/cleaned_ecommerce_oct.parquet`